In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [3]:
class EvaluationSchema(BaseModel):
    feedback: str=Field(description='Detailed feedback for the essay')
    score: int=Field(description='Score out of 10',ge=0, le=10)

In [4]:
structured_model=model.with_structured_output(EvaluationSchema)

In [5]:
essay=""" Artificial Intelligence: Transforming the Future

Artificial Intelligence (AI) is one of the most revolutionary technologies of the twenty-first century. It refers to the ability of machines and computer systems to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, decision-making, and understanding language. Over the past few decades, AI has evolved from a theoretical concept into a practical technology that affects almost every aspect of human life. From smartphones and virtual assistants to self-driving cars and medical diagnosis systems, artificial intelligence has become an integral part of modern society.

The concept of artificial intelligence dates back to the 1950s when scientists and researchers began exploring the possibility of creating machines that could mimic human thinking. Early developments were limited because of the lack of computing power and data. However, advancements in computer science, machine learning, and the availability of large datasets have accelerated the growth of AI. Today, AI systems are capable of analyzing enormous amounts of information and making predictions with remarkable accuracy.

One of the most significant applications of artificial intelligence is in healthcare. AI-powered tools help doctors diagnose diseases, analyze medical images, and recommend treatment plans. Machine learning algorithms can detect patterns in medical data that may not be visible to human experts. For example, AI systems are used to identify early signs of cancer, predict patient outcomes, and monitor chronic diseases. These innovations improve the quality of healthcare and enable medical professionals to provide faster and more accurate treatment.

Another area where AI has made a major impact is education. Intelligent learning systems provide personalized educational experiences by adapting content according to the needs and abilities of individual students. Online learning platforms use AI to recommend study materials, assess performance, and provide instant feedback. Virtual tutors and chatbots assist students in understanding complex topics and answering questions at any time. As a result, education has become more accessible and efficient for learners around the world.

Artificial intelligence has also transformed the business sector. Companies use AI to automate repetitive tasks, analyze customer behavior, and improve decision-making processes. Chatbots and virtual assistants enhance customer service by providing immediate responses to queries. AI-driven analytics help organizations understand market trends, optimize operations, and increase productivity. In industries such as manufacturing, AI-powered robots perform tasks with high precision and efficiency, reducing costs and minimizing human error.

The transportation industry is experiencing significant changes because of AI. Self-driving vehicles, intelligent traffic management systems, and predictive maintenance technologies are improving safety and efficiency. Autonomous cars use sensors, cameras, and machine learning algorithms to navigate roads and avoid obstacles. Although fully autonomous transportation is still under development, AI has the potential to reduce accidents caused by human error and create smarter transportation networks.

Artificial intelligence is also widely used in communication and entertainment. Streaming platforms recommend movies and songs based on user preferences. Social media platforms use AI algorithms to personalize content and improve user experiences. Voice assistants such as Siri, Alexa, and Google Assistant enable users to interact with devices using natural language. AI-powered translation tools help people communicate across different languages, promoting global connectivity and understanding.

Despite its numerous benefits, artificial intelligence also presents several challenges and concerns. One major issue is the impact of automation on employment. As machines become capable of performing tasks previously done by humans, certain jobs may become obsolete. Workers in industries that rely heavily on repetitive activities may face displacement. Therefore, governments and organizations must invest in education and skill development to prepare individuals for new opportunities in an AI-driven economy.

Ethical concerns are another important aspect of artificial intelligence. AI systems rely on data, and biased or inaccurate data can lead to unfair decisions. Privacy and security issues have also become significant as AI applications collect and process vast amounts of personal information. Ensuring transparency, accountability, and responsible use of AI technologies is essential to maintaining public trust. Researchers and policymakers are working to establish ethical guidelines and regulations that promote the safe and fair development of artificial intelligence.

Furthermore, there are concerns about the misuse of AI technologies. Deepfake videos, cyberattacks, and autonomous weapons are examples of how AI can be used for harmful purposes. These risks highlight the importance of international cooperation and strict regulations to prevent the abuse of artificial intelligence. Responsible innovation and continuous monitoring are necessary to ensure that AI benefits society rather than causing harm.

The future of artificial intelligence holds immense promise. Researchers are exploring advanced forms of AI that can solve complex scientific problems, improve environmental sustainability, and assist in space exploration. AI has the potential to accelerate discoveries in medicine, develop smarter cities, and address global challenges such as climate change and resource management. As technology continues to evolve, collaboration between governments, industries, researchers, and society will play a crucial role in shaping the future of AI.

In conclusion, artificial intelligence is transforming the world in unprecedented ways. Its applications in healthcare, education, business, transportation, and communication have brought significant improvements to human life. However, the rapid development of AI also raises important ethical, social, and economic challenges that must be addressed responsibly. By promoting innovation while ensuring fairness and accountability, society can harness the power of artificial intelligence to create a more efficient, connected, and prosperous future for generations to come.

"""

In [6]:
prompt="Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
structured_model.invoke(prompt)

EvaluationSchema(feedback='The essay content was not provided, so a detailed evaluation of its language quality is not possible. To improve language quality, focus on clear and concise phrasing, varied sentence structures, precise vocabulary, correct grammar, punctuation, and spelling. Ensure coherence and cohesion throughout the text. Proofreading carefully is crucial for identifying and correcting errors.', score=0)

In [7]:
structured_model.invoke(prompt).feedback


'The language quality of the essay is generally good, demonstrating a solid grasp of grammar and a varied vocabulary. Sentence structures are mostly clear and effective, contributing to the overall readability. There are occasional minor issues with subject-verb agreement and pronoun reference that, if addressed, would further enhance the precision of the writing. The essay flows well, with smooth transitions between paragraphs. To achieve a higher score, focus on refining these minor grammatical inconsistencies and exploring more sophisticated rhetorical devices to elevate the prose.'

In [8]:
structured_model.invoke(prompt).score


0

In [9]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int],operator.add]
    avg_score: float

In [17]:
def evaluate_language(state:UPSCState):

    prompt=f'Evaluate the language of the follevaluate_analysisowing essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'language_feedback':output.feedback, 'individual_scores':[output.score]}

In [18]:
def evaluate_analysis(state:UPSCState):

    prompt=f'Evaluate the analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'analysis_feedback':output.feedback, 'individual_scores':[output.score]}

In [19]:
def evaluate_thought(state:UPSCState):

    prompt=f'Evaluate the clarity of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'clarity_feedback':output.feedback, 'individual_scores':[output.score]}

In [20]:
def final_evaluation(state:UPSCState):

    ## summary feedback
    prompt=f'Based on the following feedbacks create a summarized feedback \n language feedback- {state["language_feedback"]} \n depth of analyis feedback= {state["analysis_feedback"]}\n clarity of feedback-{state["clarity_feedback"]}'
    overall_feedback=model.invoke(prompt).content

    ## avg calculate
    avg_score= sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback':overall_feedback,'avg_score':avg_score}

In [21]:
graph=StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)


graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')

graph.add_edge('final_evaluation', END)

workflow=graph.compile()

In [22]:
initial_state={
    'essay':essay
}

workflow.invoke(initial_state)

{'essay': ' Artificial Intelligence: Transforming the Future\n\nArtificial Intelligence (AI) is one of the most revolutionary technologies of the twenty-first century. It refers to the ability of machines and computer systems to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving, decision-making, and understanding language. Over the past few decades, AI has evolved from a theoretical concept into a practical technology that affects almost every aspect of human life. From smartphones and virtual assistants to self-driving cars and medical diagnosis systems, artificial intelligence has become an integral part of modern society.\n\nThe concept of artificial intelligence dates back to the 1950s when scientists and researchers began exploring the possibility of creating machines that could mimic human thinking. Early developments were limited because of the lack of computing power and data. However, advancements in computer science, machine 